In [5]:
import pandas as pd
import numpy as np

LAMBDA = 1
LOG_SCALE = 8
SCALE = 1 << LOG_SCALE
PRIME_64 = 18446744072637906947
PRIME_128 = 340282366920938463463374607429104828419
PRIME_256 = 0xFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFF98C00003
PRIME = PRIME_64

In [6]:
import os
os.system("pwd")

/workspaces/dev_rc8/nada-algebra/examples/linear_regression_training/src


0

We load the dataset here:

In [39]:
import pandas as pd

df = pd.read_csv("Student_Performance.csv")
df = df[:330]
df

,Hours Studied,Previous Scores,Extracurricular Activities,Sleep Hours,Sample Question Papers Practiced,Performance Index
0,7,99,Yes,9,1,91.0
1,4,82,No,4,2,65.0
2,8,51,Yes,7,2,45.0
3,5,52,Yes,5,2,36.0
4,7,75,No,8,5,66.0
...,...,...,...,...,...,...
325,6,62,Yes,7,6,51.0
326,6,45,No,4,0,30.0
327,1,55,Yes,8,1,29.0
328,4,41,Yes,8,2,22.0


We rescale all entries to the range $[-1, 1]$. For booleans we take -1 False, 1 True. For Strings, we take all possible values, in this case there are only two, -1 and 1.

In [40]:
a = df.to_numpy()
a[:, 2] = [1 if x == "Yes" else 0 for x in a[:, 2]]
a = ((a  - np.min(a)) / (np.max(a) - np.min(a))) * 2 - 1
# minmax = lambda x: (x - np.min(x)) / (np.max(x) - np.min(x)) * 2 - 1
# for i in [0, 1, 3, 4, 5]:
#     a[:, i] = minmax(a[:, i])
# print(a[:, 2][0])
# 
x = a[:, :-1]
y = a[:, -1]


x[0:3], y[0:3], x.shape, y.shape

(array([[-0.86, 0.98, -0.98, -0.8200000000000001, -0.98],
        [-0.92, 0.6399999999999999, -1.0, -0.92, -0.96],
        [-0.84, 0.020000000000000018, -0.98, -0.86, -0.96]], dtype=object),
 array([0.8200000000000001, 0.30000000000000004, -0.09999999999999998],
       dtype=object),
 (330, 5),
 (330,))

-1.0

In [56]:
from sklearn.linear_model import Ridge

clf = Ridge(alpha=1, fit_intercept=False, solver='lsqr')
clf.fit(x[:10, :], y[:10]), clf.coef_, clf.intercept_

(Ridge(alpha=1, fit_intercept=False, solver='lsqr'),
 array([ 0.00274544,  0.44783233, -0.00778863,  0.02378313, -0.01596796]),
 0.0)

(array([ 3.16970342,  0.9586191 , -2.97558352,  0.12067745,  0.28230238]), 0.0)

In [45]:
x_ = np.round((x * SCALE).astype(np.float64)).astype(int)[:10]
y_ = np.round((y * SCALE).astype(np.float64)).astype(int)[:10]
x_, y_

(array([[-220,  251, -251, -210, -251],
        [-236,  164, -256, -236, -246],
        [-215,    5, -251, -220, -246],
        [-230,   10, -251, -230, -246],
        [-220,  128, -256, -215, -230],
        [-241,  143, -256, -210, -225],
        [-220,  118, -251, -230, -225],
        [-215,  -26, -251, -236, -225],
        [-230,  138, -256, -215, -246],
        [-236,  200, -256, -236, -256]]),
 array([210,  77, -26, -72,  82,  56,  67, -41,  56,  97]))

In [46]:
np.where(x_ == 0)


(array([], dtype=int64), array([], dtype=int64))

In [ ]:
#x_[np.where(x_ == 0)] = x[np.where(x_ == 0)] + 1

#np.where(x_ == 0)

In [47]:
print("""program: linear_regression
inputs:
  secrets:""" )
for i, x__  in enumerate(x_):
    for j, y__ in enumerate(x__):
        print(f"""    A_{i}_{j}:
      SecretInteger: "{y__}" # round({x[i][j]} * SCALE)""")

for i, y__ in enumerate(y_):
    print(f"""    b_{i}:
      SecretInteger: "{y__}" # round({y[i]} * SCALE)""")
#print("lambda: ", LAMBDA * SCALE**2)
print(f"""  public_variables: 
    lambda_0: 
      Integer: "{LAMBDA * SCALE**2}"
      """)
print(f"""expected_outputs:
  b_0:
    SecretInteger: "0" """)
for i in range(x_.shape[1]):
    print(f"""  w_{i}:
    SecretInteger: "0" """)


program: linear_regression
inputs:
  secrets:
    A_0_0:
      SecretInteger: "-220" # round(-0.86 * SCALE)
    A_0_1:
      SecretInteger: "251" # round(0.98 * SCALE)
    A_0_2:
      SecretInteger: "-251" # round(-0.98 * SCALE)
    A_0_3:
      SecretInteger: "-210" # round(-0.8200000000000001 * SCALE)
    A_0_4:
      SecretInteger: "-251" # round(-0.98 * SCALE)
    A_1_0:
      SecretInteger: "-236" # round(-0.92 * SCALE)
    A_1_1:
      SecretInteger: "164" # round(0.6399999999999999 * SCALE)
    A_1_2:
      SecretInteger: "-256" # round(-1.0 * SCALE)
    A_1_3:
      SecretInteger: "-236" # round(-0.92 * SCALE)
    A_1_4:
      SecretInteger: "-246" # round(-0.96 * SCALE)
    A_2_0:
      SecretInteger: "-215" # round(-0.84 * SCALE)
    A_2_1:
      SecretInteger: "5" # round(0.020000000000000018 * SCALE)
    A_2_2:
      SecretInteger: "-251" # round(-0.98 * SCALE)
    A_2_3:
      SecretInteger: "-220" # round(-0.86 * SCALE)
    A_2_4:
      SecretInteger: "-246" # round(-0.9

In [48]:
w = [-894378891176476637, 3013533184205657302, 4934136027053134616, 8202649693700334710, 1519170381621038631]
b = 2642559726089895491

w_ = np.array(w) / b
w_

array([-0.33845172,  1.14038413,  1.86718051,  3.10405461,  0.57488592])

In [49]:
for i in range(x.shape[0]):
    print(f"{np.dot(w_, x[i])} == {y[i]}")

-3.5299049517431147 == 0.8200000000000001
-4.233579806498224 == 0.30000000000000004
-4.744107220512348 == -0.09999999999999998
-4.825154618790158 == -0.28
-4.1307231647162475 == 0.32000000000000006
-3.961645168316901 == 0.21999999999999997
-4.313740477831958 == 0.26
-5.021205719353279 == -0.16000000000000003
-4.106062885716821 == 0.21999999999999997
-4.096921464603981 == 0.3799999999999999
-4.020893644966584 == 0.6799999999999999
-4.204916808022261 == 0.45999999999999996
-4.714674204932702 == -0.45999999999999996
-5.045386769047097 == -0.33999999999999997
-4.053542302012132 == 0.3600000000000001
-4.436620845935984 == -0.14
-4.157261672660165 == 0.3400000000000001
-4.000723305990427 == 0.3999999999999999
-4.7408292133161725 == -0.4
-4.236754780055987 == 0.26
-3.776700768967277 == 0.41999999999999993
-3.6404002940127635 == 0.7
-4.180308679752299 == 0.45999999999999996
-4.036777689296455 == 0.1399999999999999
-4.5701122055135865 == -0.30000000000000004
-4.4634582397943285 == -0.0200000000

-0.7777777777777778